In [ ]:
# Install once if required
# %pip install tensorflow matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

tf.random.set_seed(42)
np.random.seed(42)

# Load and preprocess dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

NORMAL = 1
x_normal = x_train[y_train == NORMAL]

# Prepare test data: all normal digit-1 samples plus 150 other digits
rng = np.random.default_rng(42)
normal_idx = np.where(y_test == NORMAL)[0]
anomaly_idx = rng.choice(np.where(y_test != NORMAL)[0], size=150, replace=False)
test_idx = np.concatenate([normal_idx, anomaly_idx])
rng.shuffle(test_idx)

x_eval = x_test[test_idx]
y_eval = (y_test[test_idx] != NORMAL).astype(int)  # 0=normal, 1=anomaly

print("Normal training samples:", len(x_normal))
print("Test samples:", len(x_eval))

# Encoder
inputs = keras.Input(shape=(784,))
encoded = layers.Dense(128, activation="relu")(inputs)
encoded = layers.Dense(64, activation="relu")(encoded)
latent = layers.Dense(16, activation="relu", name="latent")(encoded)

# Decoder
decoded = layers.Dense(64, activation="relu")(latent)
decoded = layers.Dense(128, activation="relu")(decoded)
outputs = layers.Dense(784, activation="sigmoid")(decoded)

autoencoder = keras.Model(inputs, outputs, name="MNIST_Autoencoder")
autoencoder.summary()

# Compile and train using normal images only
autoencoder.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = autoencoder.fit(
    x_normal, x_normal,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    shuffle=True,
    verbose=2
)

# Calculate reconstruction error
def reconstruction_error(data):
    reconstructed = autoencoder.predict(data, verbose=0)
    return np.mean((data - reconstructed) ** 2, axis=1)

train_error = reconstruction_error(x_normal)
threshold = train_error.mean() + 2 * train_error.std()

test_error = reconstruction_error(x_eval)
predictions = (test_error > threshold).astype(int)

print("Threshold:", round(float(threshold), 5))
print("ROC-AUC:", round(float(roc_auc_score(y_eval, test_error)), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_eval, predictions, labels=[0, 1]))
print(classification_report(
    y_eval, predictions,
    labels=[0, 1],
    target_names=["normal", "anomaly"],
    zero_division=0
))

# Plot training loss and reconstruction errors
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["loss"], label="Training loss")
ax[0].plot(history.history["val_loss"], label="Validation loss")
ax[0].set_title("Autoencoder Training Loss")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].hist(test_error[y_eval == 0], bins=30, alpha=0.6, label="Normal")
ax[1].hist(test_error[y_eval == 1], bins=30, alpha=0.6, label="Anomaly")
ax[1].axvline(threshold, linestyle="--", label="Threshold")
ax[1].set_title("Reconstruction Error")
ax[1].set_xlabel("Mean Squared Error")
ax[1].legend()
plt.tight_layout()
plt.show()

# Display original and reconstructed test images
sample = x_eval[:5]
reconstructed = autoencoder.predict(sample, verbose=0)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(5):
    axes[0, i].imshow(sample[i].reshape(28, 28), cmap="gray")
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()
